# Creating Cutouts of Roman Data with Astrocut

---

## Kernel Information and Read-Only Status

To run this notebook, please select "Roman Research Nexus {VERSION}" kernel at the top right of your window. For example "Roman Research Nexus 2026.2". 

This notebook is read-only. You can run cells and make edits, but you must save changes to a different location. We recommend saving the notebook within your home directory, or to a new folder within your home (e.g. <span style="font-variant:small-caps;">file > save notebook as > my-nbs/nb.ipynb</span>). Note that a directory must exist before you attempt to add a notebook to it.

## Introduction

The [`astrocut`](https://astrocut.readthedocs.io/en/latest/) tool allows users to generate cutouts out of large astronomical images. In this tutorial, we demonstrate how to use `astrocut` to generate cutouts from a sumlated calibrated rate image product Level 2 (L2) created with Roman I-Sim.

### Disclaimer
This notebook is an introductory demonstration of `astrocut's` **ASDFCutout** class for image cutouts. The examples here highlight usage of creating a few cutouts from a single image, for demonstration purposes only.  For production use of large numbers of cutouts, please follow the **[Performance Best Practices](#Performance-Best-Practices)** guidelines below.

## Imports

The following packages are used for these reasons:

- *asdf* - To handle ASDF input and output
- *roman_datamodels* - To read Roman WFI ASDF files
- *s3fs* - To access cloud files as though they were local
- *matplotlib* - The plotting package used for demonstrating the cutout
- *astrocut* - The main package import, used to generate the cutouts from the input file
- *astropy* - To manage the output FITS data file type and handle input sky coordinates

In [ ]:
import os
import warnings
from copy import deepcopy

import asdf
import astropy.units as u
from matplotlib.patches import Rectangle
import matplotlib.pyplot as plt
import s3fs
from astrocut import get_center_pixel, ASDFCutout
from astropy.coordinates import SkyCoord
from astropy.nddata import Cutout2D
from astropy.utils.exceptions import AstropyDeprecationWarning
import roman_datamodels as rdm

# Ignore warnings from the 'astropy.wcs' module
# When creating cutouts -> WARNING: Polynomial distortion is not implemented.
warnings.filterwarnings('ignore', module='astropy.wcs')

# Ignore AstropyDeprecationWarning
# When creating cutouts -> WARNING: AstropyDeprecationWarning: The class "Fits" has been renamed to "FITS" in version 7.0.
# The old name is deprecated and may be removed in a future version. Use FITS instead. [astropy.units.format]
warnings.filterwarnings('ignore', category=AstropyDeprecationWarning)

---

## Setup

The first step of the analysis is to read the Roman WFI image data, which are stored in Advanced Scientific Data Format ([ASDF](https://roman-docs.stsci.edu/data-handbook-home/wfi-data-format/introduction-to-asdf)) files. For this example, we start with a calibrated L2 simulated image created with [Roman I-Sim](https://romanisim.readthedocs.io/en/latest/). For more information about Roman's data products, check the [Roman User Documentation](https://roman-docs.stsci.edu/).

After launch, Roman data will be available through the [Mikulski Archive for Space Telescopes (MAST)](https://archive.stsci.edu/). For testing purposes, some simulated data has been placed in a separate S3 bucket. In this notebook, we access a simulated Roman data file from the RRN S3 bucket using `s3fs`. For more information about how to access the S3 bucket, please see the [Data Discovery and Access](../data_discovery_and_access/data_discovery_and_access.ipynb) tutorial.

In [ ]:
# We open a connection to the notebooks S3 storage area and
# select our file.
fs = s3fs.S3FileSystem(anon=True)

asdf_file_uri = 'r0003201001001001004_0001_wfi01_f106_cal.asdf'
if os.path.exists(asdf_file_uri):
    with rdm.open(asdf_file_uri) as dm:
        data = deepcopy(dm.data)  # Loads the WFI data into memory
        gwcs = deepcopy(dm.meta.wcs)  # Loads the WCS information into memory
else:
    asdf_dir_uri = "s3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/"
    asdf_file_uri = asdf_dir_uri + 'r0003201001001001004_0001_wfi02_f106_cal.asdf'

    # Read the relevant data into memory
    with fs.open(asdf_file_uri, 'rb') as f:
        with asdf.open(f) as af:
            dm = rdm.open(af)
            data = deepcopy(dm.data)  # Loads the WFI data into memory
            gwcs = deepcopy(dm.meta.wcs)  # Loads the WCS information into memory

## Examine Coordinates

### GWCS

Roman ASDF images use a [Generalized World Coordinate System (GWCS)](https://gwcs.readthedocs.io/en/latest/index.html) object to handle coordinate transformations, as opposed to the [Flexible Image Transport System (FITS)](https://fits.gsfc.nasa.gov/fits_primer.html) [WCS standard](https://fits.gsfc.nasa.gov/fits_wcs.html). GWCS is more flexible than FITS WCS, and contains a compound model of transformations from detector to world coordinates. Let's examine the sky and pixel coordinates of our image:

In [ ]:
# Pring the GWCS Info:
print("=== WCS Information ===")
print(f"Input frame : {gwcs.input_frame}")
print(f"Output frame: {gwcs.output_frame}")

bb = gwcs.bounding_box
if bb is None:
    print("Bounding box : <none>")
else:
    # Iterate over intervals of bb  ModelBoundingBox
    intervals = {}
    for name, ivl in bb.intervals.items():
        intervals[name] = (ivl.lower, ivl.upper)

    # Roman WFI uses (x0, x1) = (col, row)
    x0_low, x0_high = intervals.get('x0', intervals[0])
    x1_low, x1_high = intervals.get('x1', intervals[1])

    print("\nPixel bounding box (inclusive half-pixel convention):")
    print(f"  x0 (columns) : [{x0_low:.1f}, {x0_high:.1f}]")
    print(f"  x1 (rows)    : [{x1_low:.1f}, {x1_high:.1f}]")
    print(f"  Shape        : {int(x0_high - x0_low)} × {int(x1_high - x1_low)} pixels")
    print(f"  Order        : {bb.order!r}")

# Sky footprint
footprint = gwcs.footprint()
print("\nSky footprint (RA, Dec in degrees):")
for i, (ra, dec) in enumerate(footprint):
    print(f"  Corner {i}: RA = {ra:9.5f}°, Dec = {dec:9.5f}°")

### Coordinate Analysis

We will analyze the coordintes of three points in the image.

First we will define some helper functions:

In [ ]:
# Define some helper functions
def get_sep(coord1, coord2):
    """ Compute separation between two sky coordinates """
    return coord1.separation(coord2).to(u.arcsec)


def get_pix_info(xx, coord):
    """ Print some pixel coordinate info """
    print('Pixel Coord:', xx)
    ss = gwcs.pixel_to_world(*xx)
    print(f"Computed Sky Coord RA, Dec = {ss.ra.deg:.6f}°, {ss.dec.deg:.6f}°")
    print('Separation:', get_sep(coord, ss))


def print_coord_info(coord):
    """ Print some sky coordinate info """
    print(f"Input Sky Coord: RA, Dec = {coord.ra.deg:.6f}°, {coord.dec.deg:.6f}°")
    # angular distance between known and WCS-computed position
    xx = gwcs.world_to_pixel(coord)
    get_pix_info(xx, coord)

Next, we will derive the coordinates for the center, the far upper-right corner, and the far lower-left corner in the image. Once derived we will compute the pixel coordinates, the computed sky coordinates, and the separation between the input and computed sky coordinates.

In [ ]:
# Sky coord of central pixel
print('Center pixel:')
print('-------------')
nrows, ncols = data.shape

# center pixel
center_row = nrows // 2
center_col = ncols // 2
center_coord = gwcs.pixel_to_world(center_col, center_row)

print(f"Center pixel: ({center_col}, {center_row})")
print_coord_info(center_coord)
print('\n')

# Sky coord at far edge (upper right)
print('Right upper corner pixel:')
print('-------------------------')
ss = gwcs.pixel_to_world(4077, 4077)
print_coord_info(ss)
print('\n')

# Sky coord at far edge (lower left)
print('Left lower corner pixel:')
print('------------------------')
ss = gwcs.pixel_to_world(2, 2)
print_coord_info(ss)
print('\n')

# Get computed pixel coordinates from world coordinates with get_center_pixel()
test_coord = SkyCoord('270.8809 -0.28737', unit='deg')  # test out different sky coordinates here!
print('Get the closest pixel location for a given set of coordinates:')
print('--------------------------------------------------------------')
print(f'Sky Coord: {test_coord.ra.deg:.6f}°, {test_coord.dec.deg:.6f}°')
pc = get_center_pixel(gwcs, test_coord.ra.value, test_coord.dec.value)
get_pix_info(pc, test_coord)

---
## Creating the ASDF Image Cutout

In this section, we will create cutouts of the ASDF image using two different coordinates.

First, for illustrative purposes, we plot the expected cutouts onto the original image using red squares.

In [ ]:
coords = [center_coord, test_coord]
cutout_size = 200

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Expected Cutout Positions')

for i, coord in enumerate(coords):
    pix_coord = get_center_pixel(gwcs, coord.ra.value, coord.dec.value)

    # Plot image at expected cutout position
    ax[i].imshow(data, vmin=0, vmax=5, origin='lower', cmap='viridis')

    # Plot the expected cutout with red square
    tmp = Cutout2D(data, position=pix_coord, size=cutout_size, mode='partial')
    tmp.plot_on_original(ax[i], color='red', linewidth=2)

    # Zoom in around center (FIXED: xlim = columns, ylim = rows)
    padding = 500
    ax[i].set_xlim(pix_coord[0] - padding, pix_coord[0] + padding)  # X = column
    ax[i].set_ylim(pix_coord[1] - padding, pix_coord[1] + padding)  # Y = row
    ax[i].set_xlabel('Column (pixel)')
    ax[i].set_ylabel('Row (pixel)')
    ax[i].set_title(f'Cutout around {coord.ra.value:.4f}, {coord.dec.value:.4f}')

plt.show()

Now, we will use `astrocut` to create cutouts of our data file. To make a cutout from an ASDF file or files, we use the `ASDFCutout` class.
The `ASDFCutout` class takes several parameters, including:

- `input_files`: The ASDF file or list of ASDF files from which to create cutouts.
- `coordinates`: The coordinates or a list of coordinates around which to create cutouts.
- `cutout_size`: The size of the cutout around the specified coordinates.
- `fill_value`: The value to use for pixels outside the original image when creating the cutout. Default is `np.nan`.
- `lite`: If True, the cutout will be created in "lite" mode, which means that it will only contain the data and an updated world coordinate system. If False, cutouts will be made from all applicable arrays in the input file and preserve all of the metadata from the input file. Default is True.

Below, we demonstrate how to create cutouts from our single input file and two coordinates.

In [ ]:
# Create the image cutout
asdf_cutout = ASDFCutout(asdf_file_uri, coords, cutout_size)

The resulting `ASDFCutout` object can be used to access the cutout's science data and metadata. It contains the following attributes:

- `cutouts`: an `~astropy.table.Table` containing the input file, coordinate, and `~astropy.nddata.Cutout2D` object for each cutout.
- `asdf_cutouts`: an `~astropy.table.Table` containing the input file, coordinate, and `~asdf.AsdfFile` object for each cutout.
- `fits_cutouts`: an `~astropy.table.Table` containing the input file, coordinate, and `~astropy.io.fits.HDUList` object for each cutout.

Below, let's print out the `cutouts` table.

In [ ]:
# Display the cutouts table
asdf_cutout.cutouts

We can access the cutout objects through the `cutout` column.

In [ ]:
# Check the first cutout shape and position
cutout = asdf_cutout.cutouts['cutout'][0]
print('Cutout Data Shape:', cutout.shape)
print('Cutout Original Position:', cutout.position_original)
print('Cutout Position:', cutout.position_cutout)

### Inspecting the ASDF Cutout Object

Let's inspect the `ASDFCutout` tree structure and `HDUList` object using the `info` method.

In [ ]:
# Print info about the first `AsdfFile` cutout object
cutout_asdf = asdf_cutout.asdf_cutouts['cutout'][0]
cutout_asdf.info()

Note that this `AsdfFile` object only contains a cutout array for the science data and a limited metadata dictionary. This is because our cutouts were made in `lite` mode for a more lightweight output. To create cutouts of all available arrays with full metadata, you would need to use the `full` mode when generating the cutouts by setting `lite=False`. **Note: Be aware that `full` cutouts use twice the memory and take twice as long to produce.**

In [ ]:
# Print info about the first `HDUList` cutout object
cutout_fits = asdf_cutout.fits_cutouts['cutout'][0]
cutout_fits.info()

#### Plot Cutouts in Pixel Coordinates

In this section, we'll plot each of the cutouts in pixel coordinates. The pixel coordinates are shown along the axes, and the cutout is a 200 x 200 pixel square.

In [ ]:
def plot_cutout_pixel(img, coords):
    """ Display the cutout using pixel coordinates """
    plt.figure(figsize=(6, 6))
    plt.imshow(img, vmin=0, vmax=5, origin="lower")
    plt.title(f'Cutout in Pixel Coordinates at {coords.ra.deg}, {coords.dec.deg}')
    plt.xlabel('Column (pixel)')
    plt.ylabel('Row (pixel)')
    plt.show()


# Loop through each cutout and plot it in pixel coordinates
for row in asdf_cutout.cutouts:
    cutout = row['cutout']
    cutout_data = cutout.data
    plot_cutout_pixel(cutout_data, row['coordinate'])

#### Plot Cutouts in Sky Coordinates

Now we will plot the cutout in sky coordinates using the projection defined by the WCS object.

In [ ]:
# We create a convenience function
def plot_cutout_world(img, coord, wcs):
    """ Display the cutout using world coordinate system """
    plt.figure(figsize=(6, 6))
    plt.title(f'Cutout in Sky Coordinates at {coord.ra.deg}, {coord.dec.deg}')
    plt.xticks([])  # Disable tick marks
    plt.yticks([])
    ax = plt.subplot(projection=wcs)
    ax.imshow(img, vmin=0, vmax=5, origin='lower')

    # Add grid lines
    ax.grid(color='white', ls='solid')

    # Overplot the original sky coordinate in cyan
    ax.scatter_coord(coord, s=100, edgecolor='cyan', facecolor='none')

    # Plot central pixel in white
    center_loc = img.shape[0] // 2
    ax.scatter([center_loc], [center_loc], s=100, edgecolor='white', facecolor='none')

    # Plot sky coord of central pixel in yellow
    if hasattr(wcs, 'all_pix2world'):
        # WCS case
        ra, dec = wcs.all_pix2world(center_loc, center_loc, 0)
    else:
        # GWCS case
        ra, dec = wcs(center_loc, center_loc)
    cc = SkyCoord(ra, dec, unit='deg')
    ax.scatter_coord(cc, s=50, edgecolor='yellow', facecolor='none')

    plt.show()


# Loop through each cutout and plot it in world coordinates
for row in asdf_cutout.cutouts:
    cutout = row['cutout']
    cutout_data = cutout.data
    cutout_wcs = cutout.wcs
    cutout_coord = row['coordinate']

    # Print the cutout wcs
    print(cutout_wcs, '\n')

    # Print FITS WCS position of various sky coordinates
    print("SkyCoords Info:")
    print('Input Requested', cutout_coord)

    # Plot the cutout in world coordinates
    plot_cutout_world(cutout_data, cutout_coord, cutout_wcs)

---

## Partial Image Cutouts

Sometimes, cutouts are made near the edge of Roman images. In these cases, part of the cutout may fall outside of the original image.

The `ASDFCutout` class takes an optional `fill_value` parameter for this circumstance. It describes the value that should be given to any pixels in the cutout that fall outside of the input image. The default value is `np.nan`.

To illustrate this, we will take a cutout from the lower right corner of the image. The cutout will be 300 pixels on each side.

In [ ]:
# We pick values for the edges that are greater
# than the Roman images
cutout_size = 300
edge_x, edge_y = 4000, 4000

# Convert to sky coordinates
edge_coord = gwcs.pixel_to_world(edge_x, edge_y)

# Plot the expected cutout with red square
tmp = Cutout2D(data, position=(edge_x, edge_y), size=cutout_size, mode='partial')

print("Input position:", tmp.input_position_original)
print("Center original:", tmp.center_original)
print("Valid bounding box:", tmp.bbox_original)
print("Original image shape:", data.shape)

# Plot the cutout on the original image
fig, ax = plt.subplots()
ax.imshow(data, vmin=0, vmax=5, origin='lower')

# Full requested footprint, centered on the requested position
ax.add_patch(Rectangle(
    (edge_x - cutout_size / 2, edge_y - cutout_size / 2),
    cutout_size,
    cutout_size,
    fill=False,
    edgecolor="red",
    label="Requested footprint",
))

# Astropy's valid, in-bounds portion
tmp.plot_on_original(ax=ax, color="cyan", label="Valid region")

ax.set_xlim(edge_x - 500, edge_x + 500)
ax.set_ylim(edge_y - 500, edge_y + 500)
ax.legend()
plt.show()

Notice how the top and right side of the requested footprint are outside of the image. We can assign the value of these pixels in our cutout using the `fill_value` parameter. Let's set the `fill_value` to 0.

In [ ]:
# Create the partial image cutout with fill_value = 0
asdf_cutout_partial = ASDFCutout(asdf_file_uri, edge_coord, cutout_size, fill_value=0)

# Get Cutout2D object
cutout_partial = asdf_cutout_partial.cutouts['cutout'][0]
print('Cutout Data Shape:', cutout_partial.shape)
print('Cutout Original Position:', cutout_partial.position_original)
print('Cutout Position:', cutout_partial.position_cutout)

Let's inspect the cutout object and display it.

In [ ]:
# Plot cutout in pixel coordinates
plot_cutout_pixel(cutout_partial.data, edge_coord)

Now we will plot the cutout in sky coordinates.

In [ ]:
# Print the cutout wcs
print(cutout_partial.wcs, '\n')

# Print FITS WCS position of various sky coordinates
print("SkyCoords Info:")
print('Input Requested', edge_coord)

# Plot the figure with wcs
plot_cutout_world(cutout_partial.data, edge_coord, cutout_partial.wcs)

As expected, the pixels that fall outside of the original image have been assigned a value of 0.

## Performance Best Practices

The largest performance bottleneck when making ADSF cutouts is file access, i.e. downloading the S3 file over the network, then opening and unpacking the ASDF file to extract the data.  A single ASDF cutout can take ~4-5 seconds per file.  Overall cutout time will scale with the number of files used. For interactive use within a notebook in the RRN, we recommend the following guidelines:

1. **Always use `lite=True` mode:**  `lite` cutouts (the default) contain only a subset of extensions from the original ASDF file, namely the metdata and the science pixel data. Cutouts run in `full` mode use twice as much memory and take twice as long to run.
2. **Use a smaller cutout size:** Astrocut outputs cutouts within an Astropy Table object, with one row per cutout, containing the in-memory cutout obejct.  The Table object will grow in memory with the size and number of cutouts.  Keeping the cutout size small, e.g. 25x25 pixels, will minimize this overhead, and allow for a larger number of cutouts processed.  Keep your pixel budget (n_rows x cutout_size^2) <= 100 million. 
3. **Pass in a list of all requested coordinates per call:** Astrocut only opens a file once.  Once open, all coordinates from that file can be cutout with minimal overhead. 
4. **Pre-filter your filelist**:  If able, pre-filter your filelist to those with known coordinates for your cutouts.  Astrocut is smart enough to identify which files contain which coordinates, but there is still a file-open cost on every file to perform the coordinate lookup. Astrocut will attempt to cut out every coordinate from every file. 
5. **Make memory-only cutouts**: If able, avoid writing cutouts to output files on disk.  Instead, create an in-memory-only cutout (the default). While writing cutouts to disk adds neglibible overhead per cutout, it can still scale with the number of requested cutouts, add some overhead, and potentially fill your RRN user storage. 

For use within notebooks, we recommend: `lite=True`, `cutout_size=25`, a list of coordiantes **<=1000** per file, and the number of files **<=50**. If you need to make cutouts from a larger number of files, we recommended creating a background subprocess for batch processing cutouts.  Be mindful of your credit usage, user disk storage, and RRN server or kernel interruptions. 
 
### Example notebook code:

**Example 1**
This example shows how to create a set of 25x25 cutouts from a list of 100 coordinates from a single file.  Coordinates are an evenly spaced grid. This produces a `cut.cutouts` Astropy Table of 100 cutouts.  
```
sources = ['s3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/r0003201001001001004_0001_wfi02_f106_cal.asdf']
coords = [
    '270.9325 -0.36897', '270.9191 -0.36896', '270.9057 -0.36894', '270.8924 -0.36890',
    '270.8790 -0.36886', '270.8656 -0.36880', '270.8523 -0.36874', '270.8389 -0.36866',
    '270.8256 -0.36857', '270.8122 -0.36848', '270.9325 -0.35612', '270.9191 -0.35611',
    '270.9057 -0.35609', '270.8923 -0.35605', '270.8789 -0.35601', '270.8656 -0.35595',
    '270.8522 -0.35589', '270.8388 -0.35581', '270.8255 -0.35572', '270.8121 -0.35563',
    '270.9325 -0.34324', '270.9191 -0.34323', '270.9057 -0.34320', '270.8923 -0.34317',
    '270.8789 -0.34313', '270.8655 -0.34307', '270.8521 -0.34301', '270.8387 -0.34293',
    '270.8254 -0.34285', '270.8120 -0.34275', '270.9324 -0.33032', '270.9190 -0.33031',
    '270.9056 -0.33029', '270.8922 -0.33026', '270.8788 -0.33021', '270.8654 -0.33016',
    '270.8520 -0.33010', '270.8387 -0.33003', '270.8253 -0.32994', '270.8119 -0.32985',
    '270.9324 -0.31738', '270.9190 -0.31737', '270.9056 -0.31735', '270.8922 -0.31731',
    '270.8788 -0.31727', '270.8654 -0.31722', '270.8520 -0.31716', '270.8386 -0.31709',
    '270.8252 -0.31701', '270.8118 -0.31692', '270.9324 -0.30441', '270.9190 -0.30440',
    '270.9056 -0.30437', '270.8922 -0.30434', '270.8787 -0.30430', '270.8653 -0.30425',
    '270.8519 -0.30419', '270.8385 -0.30412', '270.8251 -0.30404', '270.8117 -0.30395',
    '270.9324 -0.29141', '270.9190 -0.29139', '270.9056 -0.29137', '270.8921 -0.29134',
    '270.8787 -0.29130', '270.8653 -0.29126', '270.8518 -0.29120', '270.8384 -0.29113',
    '270.8250 -0.29105', '270.8116 -0.29096', '270.9324 -0.27838', '270.9190 -0.27837',
    '270.9055 -0.27835', '270.8921 -0.27832', '270.8786 -0.27828', '270.8652 -0.27823',
    '270.8518 -0.27817', '270.8383 -0.27810', '270.8249 -0.27803', '270.8115 -0.27794',
    '270.9324 -0.26532', '270.9189 -0.26531', '270.9055 -0.26529', '270.8920 -0.26526',
    '270.8786 -0.26522', '270.8652 -0.26518', '270.8517 -0.26512', '270.8383 -0.26505',
    '270.8248 -0.26498', '270.8114 -0.26490', '270.9324 -0.25224', '270.9189 -0.25223',
    '270.9055 -0.25221', '270.8920 -0.25218', '270.8786 -0.25214', '270.8651 -0.25210',
    '270.8516 -0.25204', '270.8382 -0.25198', '270.8248 -0.25190', '270.8113 -0.25182',
]
cut = ASDFCutout(sources, coords)
```
**Example 2**
This example shows how to pass a single list of coordinates with multiple source files.  In this example, the second file contains ~60% spatial overlap with the first file. The coordinate list contains 100 coordinates known to exist in the first file, and 50 from the second. Approximately, ~72 coordinates overlap in both files. This produces a ` cut.cutouts` Astropy Table of 222 cutouts. 
```
sources = [
    's3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/r0003201001001001004_0001_wfi02_f106_cal.asdf',
    's3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/r0003201001001001004_0002_wfi02_f106_cal.asdf',
]
coords = [
    '270.9325 -0.36897', '270.9191 -0.36896', '270.9057 -0.36894', '270.8924 -0.36890',
    '270.8790 -0.36886', '270.8656 -0.36880', '270.8523 -0.36874', '270.8389 -0.36866',
    '270.8256 -0.36857', '270.8122 -0.36848', '270.9325 -0.35612', '270.9191 -0.35611',
    '270.9057 -0.35609', '270.8923 -0.35605', '270.8789 -0.35601', '270.8656 -0.35595',
    '270.8522 -0.35589', '270.8388 -0.35581', '270.8255 -0.35572', '270.8121 -0.35563',
    '270.9325 -0.34324', '270.9191 -0.34323', '270.9057 -0.34320', '270.8923 -0.34317',
    '270.8789 -0.34313', '270.8655 -0.34307', '270.8521 -0.34301', '270.8387 -0.34293',
    '270.8254 -0.34285', '270.8120 -0.34275', '270.9324 -0.33032', '270.9190 -0.33031',
    '270.9056 -0.33029', '270.8922 -0.33026', '270.8788 -0.33021', '270.8654 -0.33016',
    '270.8520 -0.33010', '270.8387 -0.33003', '270.8253 -0.32994', '270.8119 -0.32985',
    '270.9324 -0.31738', '270.9190 -0.31737', '270.9056 -0.31735', '270.8922 -0.31731',
    '270.8788 -0.31727', '270.8654 -0.31722', '270.8520 -0.31716', '270.8386 -0.31709',
    '270.8252 -0.31701', '270.8118 -0.31692', '270.9324 -0.30441', '270.9190 -0.30440',
    '270.9056 -0.30437', '270.8922 -0.30434', '270.8787 -0.30430', '270.8653 -0.30425',
    '270.8519 -0.30419', '270.8385 -0.30412', '270.8251 -0.30404', '270.8117 -0.30395',
    '270.9324 -0.29141', '270.9190 -0.29139', '270.9056 -0.29137', '270.8921 -0.29134',
    '270.8787 -0.29130', '270.8653 -0.29126', '270.8518 -0.29120', '270.8384 -0.29113',
    '270.8250 -0.29105', '270.8116 -0.29096', '270.9324 -0.27838', '270.9190 -0.27837',
    '270.9055 -0.27835', '270.8921 -0.27832', '270.8786 -0.27828', '270.8652 -0.27823',
    '270.8518 -0.27817', '270.8383 -0.27810', '270.8249 -0.27803', '270.8115 -0.27794',
    '270.9324 -0.26532', '270.9189 -0.26531', '270.9055 -0.26529', '270.8920 -0.26526',
    '270.8786 -0.26522', '270.8652 -0.26518', '270.8517 -0.26512', '270.8383 -0.26505',
    '270.8248 -0.26498', '270.8114 -0.26490', '270.9324 -0.25224', '270.9189 -0.25223',
    '270.9055 -0.25221', '270.8920 -0.25218', '270.8786 -0.25214', '270.8651 -0.25210',
    '270.8516 -0.25204', '270.8382 -0.25198', '270.8248 -0.25190', '270.8113 -0.25182',
    '270.8842 -0.34125', '270.8708 -0.34124', '270.8574 -0.34122', '270.8440 -0.34118',
    '270.8307 -0.34114', '270.8173 -0.34108', '270.8039 -0.34101', '270.7906 -0.34094',
    '270.7772 -0.34085', '270.7639 -0.34075', '270.8841 -0.31229', '270.8707 -0.31228',
    '270.8573 -0.31226', '270.8439 -0.31222', '270.8306 -0.31218', '270.8172 -0.31213',
    '270.8038 -0.31206', '270.7904 -0.31199', '270.7770 -0.31190', '270.7637 -0.31181',
    '270.8841 -0.28318', '270.8707 -0.28316', '270.8573 -0.28314', '270.8439 -0.28311',
    '270.8304 -0.28307', '270.8170 -0.28302', '270.8036 -0.28296', '270.7902 -0.28289',
    '270.7768 -0.28281', '270.7634 -0.28272', '270.8841 -0.25392', '270.8707 -0.25390',
    '270.8572 -0.25388', '270.8438 -0.25385', '270.8303 -0.25382', '270.8169 -0.25377',
    '270.8035 -0.25371', '270.7900 -0.25364', '270.7766 -0.25356', '270.7632 -0.25348',
    '270.8841 -0.22451', '270.8706 -0.22450', '270.8571 -0.22449', '270.8437 -0.22446',
    '270.8302 -0.22442', '270.8168 -0.22438', '270.8033 -0.22432', '270.7899 -0.22426',
    '270.7764 -0.22418', '270.7630 -0.22410',
]
cut = ASDFCutout(sources, coords)
```
For coordinates that do not overlap within an image, `astrocut` will issue a `DataWarning`, e.g.
```
WARNING: DataWarning: Cutout of s3://stpubdata/roman/nexus/soc_simulations/tutorial_data/roman-2026.2/r0003201001001001004_0001_wfi02_f106_cal.asdf at <SkyCoord (ICRS): (ra, dec) in deg
    (270.7904, -0.31199)> does not overlap the image. Skipping this coordinate for this file. [astrocut.asdf_cutout]
```
To suppress these warnings, you can do the following:
```
import warnings
from astrocut import ASDFCutout
from astrocut.exceptions import DataWarning

with warnings.catch_warnings():
    warnings.simplefilter('ignore', DataWarning)
    cut2 = ASDFCutout(sources, coords)
```

---

## Additional Resources

- [Astrocut Documentation](https://astrocut.readthedocs.io)
- [Advanced Scientific Data Format Documentation](https://asdf.readthedocs.io/en/latest/)

## About this Notebook

**Authors**: Thomas Dutkiewicz, Brian Cherinka, Sam Bianco, Rosa Diaz

**Last Updated**: 2026-10-14

<table width="100%" style="border:none; border-collapse:collapse;">
  <tr style="border:none;">
    <td style="border:none; width:180px; white-space:nowrap;">
       <a href="#top" style="text-decoration:none; color:#0066cc;"> Top of page</a>
    </td>
    <td style="border:none; text-align:center;">
        <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/roman_logo.png" alt="roman_logo" width="50px">
    </td>
    <td style="border:none; text-align:right;">
       <img src="https://raw.githubusercontent.com/spacetelescope/roman_notebooks/refs/heads/main/stsci_logo2.png" width="90">
    </td>
  </tr>
</table>